# Query-based: RT-DETR-l (Q0–Q1)
Kaggle: Input — дані змагання · GPU T4 ×2 · Internet On · Secret `WANDB_API_KEY`. Запуск: Save Version → Save & Run All.

Той самий протокол, що й для YOLO (A0): train кожен 4-й кадр, imgsz 1024, 10 епох, оцінка на тому ж val.

| Ран | Що змінено | Навіщо |
|---|---|---|
| Q0 | RT-DETR-l, 300 queries (дефолт) | бейслайн query-based детектора |
| Q1 | 500 queries | у кадрі до 180 об'єктів (EDA §4): чи вистачає 300 запитів |

Кожен ран: тренування → оцінка на val → W&B → сабміт → артефакти. Якщо ран падає, черга продовжується.

## 1. Конфігурація і середовище

In [1]:
RUN_QUEUE = [
    dict(RUN_ID="Q0_rtdetr_l_1024", MODEL="rtdetr-l.pt", BASE=None, CHANGE="baseline (300 queries)", IMGSZ=1024, BATCH=8),
    dict(RUN_ID="Q1_rtdetr_l_nq500_1024", MODEL="nq500", BASE="rtdetr-l.pt", CHANGE="500 queries (scene density up to 180)", IMGSZ=1024, BATCH=8),
]
FAMILY = "query_based"
EPOCHS = 10
DEBUG  = False                     # True: 2 кліпи, 1 епоха, 50 кадрів — перевірка всієї черги за ~10 хв

PROTOCOL = dict(train_step=4, val_step=2, train_max_side=1600, seed=0,
                conf=0.001, nms_iou=0.7, max_det=500)
NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
WANDB_PROJECT = "hw2-ucu-2026"
WANDB_ENTITY  = "solchanykv-ukrainian-catholic-university"

import importlib.util, os, subprocess, sys
for pkg in ["ultralytics", "pycocotools"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import contextlib, copy, gc, io, json, pickle, random, shutil, time, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO, RTDETR, settings


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def find_root():
    for pat in ["*", "*/*", "*/*/*"]:
        for c in Path("/kaggle/input").glob(pat):
            if (c / "train" / "annotations_coco.json").exists():
                return c
    raise FileNotFoundError("Додай дані змагання в Input")


def wandb_login():
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("hw2")
    except Exception as e:
        print("W&B вимкнено:", e)
        return False
    os.environ.update(WANDB_API_KEY=key, WANDB_PROJECT=WANDB_PROJECT, WANDB_ENTITY=WANDB_ENTITY)
    return wandb.login(key=key, relogin=True)


seed_everything(PROTOCOL["seed"])
ROOT = find_root()
WORK = Path("/kaggle/working"); DS = WORK / "ds"; ART = WORK / "artifacts"; ART.mkdir(exist_ok=True)
MAX_CLIPS = 2 if DEBUG else None
if DEBUG:
    EPOCHS = 1

assert torch.cuda.device_count() == 2, "Accelerator: GPU T4 x2"
USE_WANDB = wandb_login()
settings.update({"wandb": USE_WANDB})
print("черга:", [r["RUN_ID"] for r in RUN_QUEUE], f"| epochs {EPOCHS} | debug {DEBUG}")
print(f"дані: {ROOT}\nultralytics {ultralytics.__version__} | torch {torch.__version__} | "
      f"GPU: {torch.cuda.get_device_name(0)} x{torch.cuda.device_count()} | W&B: {USE_WANDB}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: solchanykv (solchanykv-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
черга: ['Q0_rtdetr_l_1024', 'Q1_rtdetr_l_nq500_1024'] | epochs 10 | debug False
дані: /kaggle/input/competitions/multiple-object-video-de

## 2. Датасет за протоколом
train — кожен 4-й кадр, ignore замальовано сірим, GT під маскою видалено, ресайз до 1600 px; val — кожен 2-й кадр, оригінали.

In [2]:
GRAY = 114


def frame_key(path):
    p = Path(path)
    return p.parent.name, p.stem


def read_yolo(path, W, H):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return np.zeros((0, 5))
    a = np.loadtxt(p, ndmin=2); out = a.copy()
    out[:, 1] = (a[:, 1] - a[:, 3] / 2) * W; out[:, 2] = (a[:, 2] - a[:, 4] / 2) * H
    out[:, 3] = a[:, 3] * W;                 out[:, 4] = a[:, 4] * H
    return out                                          # cls, x, y, w, h (px)


def read_ignore(path, W, H):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return np.zeros((0, 4))
    a = np.loadtxt(p, ndmin=2)[:, -4:]
    return np.c_[(a[:, 0] - a[:, 2] / 2) * W, (a[:, 1] - a[:, 3] / 2) * H, a[:, 2] * W, a[:, 3] * H]


def frac_inside(boxes, regions):
    if len(boxes) == 0 or len(regions) == 0:
        return np.zeros(len(boxes))
    a = np.c_[boxes[:, :2], boxes[:, :2] + boxes[:, 2:4]]
    b = np.c_[regions[:, :2], regions[:, :2] + regions[:, 2:4]]
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    return (iw * ih / np.maximum(boxes[:, 2] * boxes[:, 3], 1e-9)[:, None]).max(1)


def list_frames(split, step=1, max_clips=None):
    clips = sorted(p for p in (ROOT / split / "images").iterdir() if p.is_dir())
    return [f for c in clips[:max_clips] for f in sorted(c.glob("*.jpg"))[::step]]


def _process(f, split, mask, max_side, drop_thr):
    clip, stem = f.parent.name, f.stem
    dst = DS / split / "images" / clip / f.name
    lbl = ROOT / split / "labels" / clip / f"{stem}.txt"
    ign = ROOT / split / "ignore" / clip / f"{stem}.txt"
    lines = lbl.read_text().strip().splitlines() if lbl.exists() else []
    n_drop = 0
    need_mask = mask and ign.exists() and ign.stat().st_size > 0
    if need_mask or max_side:
        img = cv2.imread(str(f)); H, W = img.shape[:2]
        if need_mask:
            reg = read_ignore(ign, W, H)
            for x, y, w, h in reg:
                img[max(0, int(y)):int(np.ceil(y + h)), max(0, int(x)):int(np.ceil(x + w))] = GRAY
            if lines:
                keep = frac_inside(read_yolo(lbl, W, H)[:, 1:], reg) <= drop_thr
                n_drop = int((~keep).sum()); lines = [l for l, k in zip(lines, keep) if k]
        if max_side and max(H, W) > max_side:
            s = max_side / max(H, W)
            img = cv2.resize(img, (round(W * s), round(H * s)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    elif not dst.exists():
        dst.symlink_to(f)
    (DS / split / "labels" / clip / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
    return n_drop


def build_split(split, step, mask=False, max_side=None, drop_thr=0.5, max_clips=None):
    frames = list_frames(split, step, max_clips)
    for clip in {f.parent.name for f in frames}:
        (DS / split / "images" / clip).mkdir(parents=True, exist_ok=True)
        (DS / split / "labels" / clip).mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(8) as ex:
        drops = list(ex.map(lambda f: _process(f, split, mask, max_side, drop_thr), frames))
    print(f"{split}: {len(frames)} кадрів, видалено GT під ignore: {sum(drops)}")


if not (DS / "data.yaml").exists():
    t0 = time.time()
    build_split("train", PROTOCOL["train_step"], mask=True, max_side=PROTOCOL["train_max_side"], max_clips=MAX_CLIPS)
    build_split("val", PROTOCOL["val_step"], mask=False, max_clips=MAX_CLIPS)
    (DS / "data.yaml").write_text(f"path: {DS}\ntrain: train/images\nval: val/images\nnames:\n"
                                  + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
    print(f"зібрано за {time.time() - t0:.0f} с")

VAL = list_frames("val", PROTOCOL["val_step"], MAX_CLIPS)
test_df = pd.read_csv(ROOT / "test_images.csv")
TEST = [ROOT / f for f in test_df.file_name]
if DEBUG:
    VAL, TEST = VAL[:50], TEST[:50]
print(f"val: {len(VAL)} | test: {len(TEST)}")

train: 6072 кадрів, видалено GT під ignore: 3069
val: 1425 кадрів, видалено GT під ignore: 0
зібрано за 114 с
val: 1425 | test: 3811


## 3. Оцінка (COCOeval з ignore, ≤500 детекцій) і її перевірка: GT як предикти → mAP = 1.0

In [3]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


class Evaluator:
    """COCOeval по кадрах VAL: ignore через iscrowd=1, не більше 500 детекцій на кадр, AP по класах."""

    def __init__(self, gt_json, image_paths):
        with contextlib.redirect_stdout(io.StringIO()):
            self.gt = COCO(str(gt_json))
        self.key2id = {frame_key(i["file_name"]): i["id"] for i in self.gt.dataset["images"]}
        self.keys = [frame_key(p) for p in image_paths]
        self.img_ids = [self.key2id[k] for k in self.keys]

    def to_coco(self, preds):
        out = []
        for k, (xyxy, s, c) in preds.items():
            if k not in self.key2id:
                continue
            for i in np.argsort(-np.asarray(s, np.float32))[:PROTOCOL["max_det"]]:
                x1, y1, x2, y2 = map(float, xyxy[i])
                out.append(dict(image_id=self.key2id[k], category_id=int(c[i]) + 1,
                                bbox=[x1, y1, x2 - x1, y2 - y1], score=float(s[i])))
        return out

    def evaluate(self, preds):
        dets = self.to_coco(preds)
        if not dets:
            return dict(mAP=0.0)
        with contextlib.redirect_stdout(io.StringIO()):
            E = COCOeval(self.gt, self.gt.loadRes(dets), "bbox")
            E.params.imgIds = self.img_ids
            E.params.maxDets = [1, 100, PROTOCOL["max_det"]]
            E.evaluate(); E.accumulate(); E.summarize()
        s, prec = E.stats, E.eval["precision"]          # precision: [IoU, recall, class, area, maxDets]
        p_all = prec[:, :, :, 0, -1]                     # pycocotools рахує stats[0] при maxDets=100 — беремо 500
        m = dict(mAP=p_all[p_all > -1].mean(), AP50=s[1], AP75=s[2], AP_small=s[3], AP_medium=s[4],
                 AP_large=s[5], AR500=s[8])
        for k, n in enumerate(NAMES):
            p = prec[:, :, k, 0, -1]
            m[f"AP_{n}"] = p[p > -1].mean() if (p > -1).any() else np.nan
        return {k: float(v) for k, v in m.items()}


ev = Evaluator(ROOT / "val" / "annotations_coco.json", VAL)
gt_preds = {}
for k, iid in zip(ev.keys, ev.img_ids):
    anns = [a for a in ev.gt.imgToAnns[iid] if not a.get("iscrowd", 0)]
    b = np.array([a["bbox"] for a in anns], np.float32).reshape(-1, 4); b[:, 2:] += b[:, :2]
    gt_preds[k] = (b, np.ones(len(b), np.float32), np.array([a["category_id"] - 1 for a in anns], np.int8))
sanity = ev.evaluate(gt_preds)
print({k: round(v, 4) for k, v in sanity.items() if k in ("mAP", "AP50", "AP_small")})
assert sanity["mAP"] > 0.99, "оцінка рахує неправильно"
del gt_preds

{'mAP': 1.0, 'AP50': 1.0, 'AP_small': 1.0}


## 4. RT-DETR-l з 500 queries: той самий конфіг Ultralytics, змінено лише `nq` у декодері

In [4]:
cfg_src = Path(ultralytics.__file__).parent / "cfg" / "models" / "rt-detr" / "rtdetr-l.yaml"
y = cfg_src.read_text()
assert "RTDETRDecoder, [nc]]" in y
(WORK / "rtdetr-l-nq500.yaml").write_text(y.replace("RTDETRDecoder, [nc]]", "RTDETRDecoder, [nc, 256, 500]]"))

m = RTDETR(str(WORK / "rtdetr-l-nq500.yaml")); m.load("rtdetr-l.pt")
print("num_queries:", m.model.model[-1].num_queries); m.info(); del m

WARNING ⚠️ no model scale passed. Assuming scale='l'.
Transferred 941/941 items from pretrained weights
num_queries: 500
rtdetr-l-nq500 summary: 470 layers, 32,970,476 parameters, 32,970,476 gradients, 114.2 GFLOPs


## 5. Функції одного рану: тренування → оцінка → W&B → сабміт → артефакти

In [5]:
import logging
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)


def predict_ultra(model, paths, imgsz, batch=8, chunk=64):
    preds = {}
    paths = [str(p) for p in paths]
    for s in range(0, len(paths), chunk):
        for r in model.predict(source=paths[s:s + chunk], imgsz=imgsz, conf=PROTOCOL["conf"],
                               iou=PROTOCOL["nms_iou"], max_det=PROTOCOL["max_det"], half=True,
                               batch=batch, stream=True, verbose=False, device=0):
            b = r.boxes
            preds[frame_key(r.path)] = (b.xyxy.cpu().numpy().astype(np.float32),
                                        b.conf.cpu().numpy().astype(np.float32),
                                        b.cls.cpu().numpy().astype(np.int8))
        gc.collect(); torch.cuda.empty_cache()
    return preds


def measure_fps(predict_fn, paths, n=100, warmup=10):
    paths = list(paths)[: n + warmup]
    for p in paths[:warmup]:
        predict_fn(p)
    torch.cuda.synchronize(); t = time.perf_counter()
    for p in paths[warmup:]:
        predict_fn(p)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t) / max(1, len(paths) - warmup) * 1000
    return dict(ms_per_img=ms, FPS=1000 / ms)



def boxes_image(path, pred, conf=0.3, max_side=1280):
    import wandb
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    s = min(1.0, max_side / max(H, W))
    if s < 1:
        img = cv2.resize(img, (round(W * s), round(H * s)), interpolation=cv2.INTER_AREA)
    clip, stem = frame_key(path)
    cl = dict(enumerate(NAMES))
    gt = read_yolo(ROOT / "val" / "labels" / clip / f"{stem}.txt", W, H)
    xyxy, sc, c = pred
    keep = sc >= conf
    box = lambda x1, y1, x2, y2: {"minX": float(x1) * s, "minY": float(y1) * s, "maxX": float(x2) * s, "maxY": float(y2) * s}
    boxes = {
        "ground_truth": {"class_labels": cl, "box_data": [
            {"position": box(x, y, x + w, y + h), "domain": "pixel", "class_id": int(k)} for k, x, y, w, h in gt]},
        "predictions": {"class_labels": cl, "box_data": [
            {"position": box(*b), "domain": "pixel", "class_id": int(k), "scores": {"conf": float(v)}}
            for b, v, k in zip(xyxy[keep], sc[keep], c[keep])]},
    }
    return wandb.Image(img, boxes=boxes, caption=f"{clip}/{stem}")



def write_submission(preds, path):
    rows = []
    for iid, fn in zip(test_df.image_id, test_df.file_name):
        k = frame_key(fn)
        if k not in preds:
            rows.append((iid, "")); continue
        xyxy, s, c = preds[k]
        o = np.argsort(-s)[:PROTOCOL["max_det"]]
        rows.append((iid, " ".join(f"{int(c[i])} {s[i]:.4f} {xyxy[i, 0]:.1f} {xyxy[i, 1]:.1f} "
                                   f"{xyxy[i, 2] - xyxy[i, 0]:.1f} {xyxy[i, 3] - xyxy[i, 1]:.1f}" for i in o)))
    df = pd.DataFrame(rows, columns=["image_id", "PredictionString"])
    df.to_csv(path, index=False)
    chk = pd.read_csv(path, keep_default_na=False)
    assert len(chk) == len(test_df) and set(chk.image_id) == set(test_df.image_id)
    n = chk.PredictionString.str.split().str.len().fillna(0)
    assert (n % 6 == 0).all() and (n <= 6 * PROTOCOL["max_det"]).all()
    print(f"сабміт: {path} | кадрів з предиктами: {(n > 0).sum()} / {len(chk)}")




def run_one(cfg):
    run_id, imgsz = cfg["RUN_ID"], cfg["IMGSZ"]
    out = ART / run_id; out.mkdir(exist_ok=True)
    print(f"\n{'=' * 30} {run_id}: {cfg['CHANGE']} {'=' * 30}")

    model = RTDETR(str(WORK / "rtdetr-l-nq500.yaml") if cfg["MODEL"] == "nq500" else cfg["MODEL"])
    if cfg["BASE"]:
        model.load(cfg["BASE"])
    t0 = time.time()
    model.train(data=str(DS / "data.yaml"), epochs=EPOCHS, imgsz=imgsz, batch=cfg["BATCH"], device=[0, 1],
                workers=4, cache=False, seed=PROTOCOL["seed"], deterministic=True, cos_lr=True,
                close_mosaic=2, amp=True, project=WANDB_PROJECT, name=run_id, exist_ok=True, plots=True,
                **cfg.get("TRAIN_KW", {}))
    train_min = (time.time() - t0) / 60
    del model; gc.collect(); torch.cuda.empty_cache()

    run_dir = max((p.parent.parent for p in WORK.rglob("weights/best.pt") if p.parent.parent.name == run_id),
                  key=lambda d: d.stat().st_mtime)
    shutil.copy(run_dir / "weights" / "best.pt", out / "best.pt")
    for f in ["results.csv", "args.yaml", "results.png", "confusion_matrix_normalized.png"]:
        if (run_dir / f).exists():
            shutil.copy(run_dir / f, out / f)

    model = RTDETR(str(out / "best.pt"))
    val_preds = predict_ultra(model, VAL, imgsz)
    metrics = ev.evaluate(val_preds)
    metrics.update(measure_fps(lambda p: model.predict(p, imgsz=imgsz, half=True, conf=PROTOCOL["conf"],
                                                       max_det=PROTOCOL["max_det"], verbose=False, device=0), VAL))
    metrics["train_min"] = train_min

    if USE_WANDB:
        import wandb
        per_class = pd.Series({n: metrics[f"AP_{n}"] for n in NAMES}, name="AP")
        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=f"{run_id}_eval", group=FAMILY,
                         job_type="eval", config=dict(PROTOCOL, **cfg, epochs=EPOCHS, debug=DEBUG))
        run.summary.update(metrics)
        run.log({"per_class_AP": wandb.Table(dataframe=per_class.reset_index().rename(columns={"index": "class"}))})
        idx = np.linspace(0, len(VAL) - 1, 12).astype(int)
        run.log({"val_predictions": [boxes_image(VAL[i], val_preds[frame_key(VAL[i])]) for i in idx]})
        run.finish()

    test_preds = predict_ultra(model, TEST, imgsz)
    write_submission(test_preds, out / f"submission_{run_id}.csv")
    for tag, p in [("val", val_preds), ("test", test_preds)]:
        pickle.dump({k: (b, s.astype(np.float16), c) for k, (b, s, c) in p.items()}, open(out / f"{tag}_preds.pkl", "wb"))
    json.dump(dict(run_id=run_id, **cfg, epochs=EPOCHS, debug=DEBUG, protocol=PROTOCOL,
                   ultralytics=ultralytics.__version__, **metrics), open(out / "metrics.json", "w"), indent=1)

    model_name = "rtdetr-l-nq500" if cfg["MODEL"] == "nq500" else cfg["MODEL"].replace(".pt", "")
    row = [run_id, FAMILY, model_name, cfg["CHANGE"], imgsz, EPOCHS] + \
          [f"{metrics[k]:.4f}" for k in ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]] + \
          [f"{metrics['FPS']:.1f}", "", f"{run_id}_eval", "DEBUG" if DEBUG else ""]

    del model, val_preds, test_preds; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(run_dir, ignore_errors=True)
    print(f"{run_id}: mAP {metrics['mAP']:.4f} | AP_small {metrics['AP_small']:.4f} | тренування {train_min:.0f} хв")
    return metrics, ",".join(map(str, row))

## 6. Черга ранів

In [6]:
import logging
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)

results = {}
for cfg in RUN_QUEUE:
    try:
        results[cfg["RUN_ID"]] = run_one(cfg)
    except Exception:
        print(f"!!! {cfg['RUN_ID']} впав:")
        traceback.print_exc()
    gc.collect(); torch.cuda.empty_cache()


============================== Q0_rtdetr_l_1024: baseline (300 queries) ==============================
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3263.4±1561.3 MB/s, size: 479.9 KB)
train: Scanning /kaggle/working/ds/train/labels/uav0000013_00000_v... 6072 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6072/6072 1.0Kit/s 6.0s
train: New cache created: /kaggle/working/ds/train/labels/uav0000013_00000_v.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 42.8±27.5 MB/s, size: 963.5 KB). Use local stora

## 7. Підсумок: метрики всіх ранів і рядки для `experiments.csv`

In [7]:
main = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "FPS", "train_min"]
display(pd.DataFrame({k: {m: v[0][m] for m in main} for k, v in results.items()}).T.round(4))
display(pd.DataFrame({k: {n: v[0][f"AP_{n}"] for n in NAMES} for k, v in results.items()}).T.round(4))
print("\nрядки для code/results/experiments.csv:")
for _, (_, row) in results.items():
    print(row)

shutil.rmtree(WORK / "wandb", ignore_errors=True)
for f in WORK.glob("*.pt"):
    f.unlink()
print("\nOutput:", sorted(p.name for p in ART.iterdir()))

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR500,FPS,train_min
Q0_rtdetr_l_1024,0.2683,0.5135,0.2450,0.1389,0.2310,0.4562,0.4106,15.1914,86.0959
Q1_rtdetr_l_nq500_1024,0.2613,0.4997,0.2409,0.1245,0.2255,0.4371,0.4136,15.1507,90.0488


,person,bicycle,car,van,truck,tricycle,bus,motor
Q0_rtdetr_l_1024,0.3029,0.2192,0.4472,0.2844,0.2778,0.2654,0.2292,0.1206
Q1_rtdetr_l_nq500_1024,0.2942,0.2118,0.4473,0.2696,0.2437,0.2363,0.2586,0.1287



рядки для code/results/experiments.csv:
Q0_rtdetr_l_1024,query_based,rtdetr-l,baseline (300 queries),1024,10,0.2683,0.5135,0.2450,0.1389,0.2310,0.4562,15.2,,Q0_rtdetr_l_1024_eval,
Q1_rtdetr_l_nq500_1024,query_based,rtdetr-l-nq500,500 queries (scene density up to 180),1024,10,0.2613,0.4997,0.2409,0.1245,0.2255,0.4371,15.2,,Q1_rtdetr_l_nq500_1024_eval,

Output: ['Q0_rtdetr_l_1024', 'Q1_rtdetr_l_nq500_1024']
